# ACB - spreadsheet to database
## Ana Carolina Brandão, Abril 2026

Este script serve para importar automaticamente para a base de dados os dados existentes numa Google Spreadsheet.

O objetivo é permitir atualizar as tabelas da base de dados a partir de dados previamente consultados e validados numa spreadsheet. Para isso, o script autentica-se no Google Sheets, lê as folhas da spreadsheet e escreve os respetivos dados nas bases de dados configuradas, incluindo Aiven e CrateDB.

As credenciais são carregadas através de secrets do Colab, usando dinamicamente as iniciais do nome do notebook para identificar o utilizador.


In [1]:
## imports
##

import json
import os
import socket
import requests
import re

current_env = os.environ.get('CONDA_DEFAULT_ENV')
print("current_env:", current_env)

if current_env is None:
    !pip install pymysql --quiet
    !pip install crate --quiet
    !pip install clts_pcp --quiet
    !pip install pandas --quiet
    !pip install gspread --quiet
    !pip install google-auth --quiet

    from google.colab import userdata
    from google.colab import auth
    from google.auth import default

import pymysql
from crate import client as crate_client
import pandas as pd
import clts_pcp as clts
import gspread

print("... done.")

current_env: None
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.5 MB/s eta 0:00:00
... done.


In [2]:
## Context gathering
##

tstart = clts.getts()

DEFAULT_PARAMS = {
    "verbose": True,
    "timeout": 20,
    "spreadsheet_url": "https://docs.google.com/spreadsheets/d/1FTPlF_maNoTUl6kmMkqHexZvVgWR6VCvnRrBw0jJ16o/edit?gid=2007178023#gid=2007178023",
    "spreadsheet_name": "Confirmed",
    "truncate_before_insert": True,
    "create_tables_if_missing": True,
    "drop_empty_rows": True
}

verbose = DEFAULT_PARAMS["verbose"]
timeout = DEFAULT_PARAMS["timeout"]
spreadsheet_url = DEFAULT_PARAMS["spreadsheet_url"]
spreadsheet_name = DEFAULT_PARAMS["spreadsheet_name"]
truncate_before_insert = DEFAULT_PARAMS["truncate_before_insert"]
create_tables_if_missing = DEFAULT_PARAMS["create_tables_if_missing"]
drop_empty_rows = DEFAULT_PARAMS["drop_empty_rows"]

hostname = socket.gethostname()

try:
    ip = requests.get("https://api.ipify.org", timeout=5).text
except:
    ip = "unknown"

print("Server name:", hostname, "Public IP Address:", ip)

if "__file__" in globals():
    enviro = "airflow/linux"
    script = os.path.basename(__file__)
    parts = __file__.replace("\\", "/").split("/")
    channel = parts[-2] if len(parts) >= 2 else "unknown"
else:
    enviro = "jupyter"
    channel = "colab"
    script = requests.get("http://172.28.0.12:9000/api/sessions").json()[0]["name"]

# Recupera as três primeiras letras do nome do ficheiro
# Exemplo: ACB_spreadsheet_to_database_Teste.ipynb -> acb
match = re.match(r"([A-Za-z]{3})", script)

if not match:
    raise ValueError(f"Não foi possível identificar o utilizador a partir do nome do ficheiro: {script}")

user = match.group(1).lower()

context = f"{hostname} ({ip}) | {user} | {channel} | {script}"
clts.setcontext(context)

if verbose:
    print("script:", script)
    print("user:", user)
    print("context:", context)


Server name: bca21bfc18d1 Public IP Address: 34.106.14.16
script: ACB-spreadsheet_to_database_Teste.ipynb
user: acb
context: bca21bfc18d1 (34.106.14.16) | acb | colab | ACB-spreadsheet_to_database_Teste.ipynb


In [3]:
## Ler secrets das bases de dados
##

aiven_secret_name = f"{user}-d5hive-aiven-super-1.json"
cratedb_secret_name = f"{user}-d5hive-cratedb-super-1.json"

aiven_json = userdata.get(aiven_secret_name)
cratedb_json = userdata.get(cratedb_secret_name)

if aiven_json is None:
    raise ValueError(f"Secret não encontrado no Colab: {aiven_secret_name}")

if cratedb_json is None:
    raise ValueError(f"Secret não encontrado no Colab: {cratedb_secret_name}")

aiven_creds = json.loads(aiven_json)
cratedb_creds = json.loads(cratedb_json)

DBS = [
    {
        "nome": "aiven",
        "tipo": "mysql",
        "enabled": True,
        "host": aiven_creds["dest_host"],
        "port": int(aiven_creds["port"]),
        "database": aiven_creds["database"],
        "user": aiven_creds["username"],
        "password": aiven_creds["password"],
    },
    {
        "nome": "cratedb",
        "tipo": "cratedb",
        "enabled": True,
        "host": cratedb_creds["host"],
        "port": int(cratedb_creds["port"]),
        "user": cratedb_creds["user"],
        "password": cratedb_creds["password"],
        "schema": cratedb_creds.get("schema", "doc"),
    }
]

print("Secrets das bases de dados carregados com sucesso.")
print("Secret Aiven usado:", aiven_secret_name)
print("Secret CrateDB usado:", cratedb_secret_name)

for db in DBS:
    print("-", db["nome"], "| tipo =", db["tipo"], "| enabled =", db["enabled"])

Secrets das bases de dados carregados com sucesso.
Secret Aiven usado: acb-d5hive-aiven-super-1.json
Secret CrateDB usado: acb-d5hive-cratedb-super-1.json
- aiven | tipo = mysql | enabled = True
- cratedb | tipo = cratedb | enabled = True


In [4]:
## Autenticação Google
##

def autenticar_google():
    try:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)

        print("Autenticação Google com sucesso!")
        clts.elapt["Google authentication successful ✅"] = clts.deltat(tstart)
        return gc

    except Exception as e:
        print("Erro na autenticação Google:", e)
        clts.elapt[f"Google authentication error ❌: {e}"] = clts.deltat(tstart)
        return None

In [5]:
## Ligação às bases de dados
##

def obter_endpoint_cratedb(cfg):
    host = cfg.get("host")
    port = cfg.get("port", 4200)

    if host and str(host).startswith(("http://", "https://")):
        return host if f":{port}" in str(host) else f"{host}:{port}"

    return f"https://{host}:{port}"


def ligar_bd(cfg):
    try:
        if cfg["tipo"] == "mysql":
            connection = pymysql.connect(
                host=cfg["host"],
                port=cfg["port"],
                db=cfg["database"],
                user=cfg["user"],
                password=cfg["password"],
                cursorclass=pymysql.cursors.DictCursor,
                charset="utf8mb4",
                connect_timeout=timeout,
                read_timeout=timeout,
                write_timeout=timeout,
                autocommit=True
            )

        elif cfg["tipo"] == "cratedb":
            endpoint = obter_endpoint_cratedb(cfg)

            connection = crate_client.connect(
                endpoint,
                username=cfg["user"],
                password=cfg["password"],
                schema=cfg.get("schema", "doc"),
                timeout=timeout,
                error_trace=True
            )

        else:
            raise Exception(f"Tipo de base de dados não suportado: {cfg['tipo']}")

        print(f"Ligação à base de dados com sucesso! [{cfg['nome']}]")
        return connection

    except Exception as e:
        print(f"Erro na ligação à base de dados [{cfg['nome']}]:", e)
        return None


def ligar_bases():
    ligacoes = []

    for cfg in DBS:
        if not cfg.get("enabled", True):
            print(f"Base ignorada [{cfg['nome']}] porque está desativada.")
            continue

        conn = ligar_bd(cfg)
        if conn is not None:
            ligacoes.append({
                "cfg": cfg,
                "conn": conn
            })

    return ligacoes

In [6]:
## Abrir Google Spreadsheet
##

def abrir_spreadsheet(gc):
    try:
        if spreadsheet_url:
            sh = gc.open_by_url(spreadsheet_url)
        elif spreadsheet_name:
            sh = gc.open(spreadsheet_name)
        else:
            raise Exception("Define spreadsheet_url ou spreadsheet_name.")

        print(f"Spreadsheet aberta com sucesso: {sh.title}")
        return sh

    except Exception as e:
        print("Erro ao abrir spreadsheet:", e)
        return None

In [7]:
## Normalização de nomes e dados
##

def limpar_nome_tabela(nome):
    nome = str(nome).strip().lower()
    nome = re.sub(r'\s+', '_', nome)
    nome = re.sub(r'[^a-zA-Z0-9_]', '_', nome)
    if not nome:
        nome = "tabela_sem_nome"
    return nome


def limpar_nome_coluna(nome):
    nome = str(nome).strip().lower()
    nome = re.sub(r'\s+', '_', nome)
    nome = re.sub(r'[^a-zA-Z0-9_]', '_', nome)
    if not nome:
        nome = "coluna_sem_nome"
    return nome


def normalizar_dataframe(df):
    df = df.copy()

    df.columns = [limpar_nome_coluna(col) for col in df.columns]

    df = df.replace(r'^\s*$', None, regex=True)

    if drop_empty_rows:
        df = df.dropna(axis=0, how="all")

    df = df.astype(object)
    df = df.where(pd.notnull(df), None)
    df = df.map(lambda x: str(x).strip() if x is not None else None)

    return df

In [8]:
## Construção de SQL para MySQL e CrateDB
##

def quote_ident(nome, cfg):
    nome = str(nome)

    if cfg["tipo"] == "mysql":
        return f"`{nome}`"

    return f'"{nome}"'

def nome_tabela_sql(nome_tabela, cfg):
    tabela = quote_ident(nome_tabela, cfg)

    if cfg["tipo"] == "cratedb":
        schema = cfg.get("schema", "doc")
        return f'{quote_ident(schema, cfg)}.{tabela}'

    return tabela

def placeholder_sql(cfg):
    return "%s" if cfg["tipo"] == "mysql" else "?"

def tipo_texto(cfg):
    if cfg["tipo"] == "mysql":
        return "TEXT NULL"
    return "TEXT"

def rows_para_dataframe(rows, cursor):
    if not rows:
        return pd.DataFrame()

    if isinstance(rows[0], dict):
        return pd.DataFrame(rows)

    colunas = []
    for desc in cursor.description or []:
        if isinstance(desc, (list, tuple)):
            colunas.append(desc[0])
        else:
            colunas.append(getattr(desc, "name", str(desc)))

    return pd.DataFrame(rows, columns=colunas)

In [9]:
## Converter folha da spreadsheet em DataFrame
##

def worksheet_para_dataframe(worksheet):
    try:
        valores = worksheet.get_all_values()

        if not valores:
            return pd.DataFrame()

        headers = valores[0]

        if not headers:
            return pd.DataFrame()

        num_cols = len(headers)

        dados = []
        for linha in valores[1:]:
            linha = linha[:num_cols] + [""] * max(0, num_cols - len(linha))
            dados.append(linha)

        df = pd.DataFrame(dados, columns=headers)

        df = df.replace("", None)
        df = normalizar_dataframe(df)

        return df

    except Exception as e:
        print(f"Erro ao ler worksheet {worksheet.title}: {e}")
        return None

In [10]:
## Criar ou atualizar estrutura da tabela
##

def criar_tabela_se_nao_existir(conn, nome_tabela, df, cfg):
    try:
        if df is None:
            print(f"Tabela {nome_tabela}: DataFrame inválido.")
            return False

        if len(df.columns) == 0:
            print(f"Tabela {nome_tabela}: sem colunas para criar.")
            return False

        cursor = conn.cursor()

        usar_id_como_pk = (
            "id" in df.columns
            and df["id"].notna().all()
            and df["id"].astype(str).nunique() == len(df)
        )

        colunas_sql = []

        if usar_id_como_pk:
            id_numerico = pd.to_numeric(df["id"], errors="coerce").notna().all()

            for col in df.columns:
                col_sql = quote_ident(col, cfg)

                if col == "id":
                    if id_numerico:
                        colunas_sql.append(f"{col_sql} BIGINT NOT NULL")
                    else:
                        if cfg["tipo"] == "mysql":
                            colunas_sql.append(f"{col_sql} VARCHAR(255) NOT NULL")
                        else:
                            colunas_sql.append(f"{col_sql} TEXT NOT NULL")
                else:
                    colunas_sql.append(f"{col_sql} {tipo_texto(cfg)}")

            pk_sql = f"PRIMARY KEY ({quote_ident('id', cfg)})"

        else:
            if cfg["tipo"] == "mysql":
                colunas_sql.append(f"{quote_ident('row_id', cfg)} BIGINT NOT NULL AUTO_INCREMENT")
                pk_sql = f"PRIMARY KEY ({quote_ident('row_id', cfg)})"
            else:
                colunas_sql.append(f"{quote_ident('row_id', cfg)} TEXT DEFAULT gen_random_text_uuid() PRIMARY KEY")
                pk_sql = None

            for col in df.columns:
                colunas_sql.append(f"{quote_ident(col, cfg)} {tipo_texto(cfg)}")

        definicoes = colunas_sql + ([pk_sql] if pk_sql else [])

        query = f"CREATE TABLE IF NOT EXISTS {nome_tabela_sql(nome_tabela, cfg)} ({', '.join(definicoes)})"

        if cfg["tipo"] == "mysql":
            query += " CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"

        query += ";"

        cursor.execute(query)
        print(f"Tabela verificada/criada: {nome_tabela}")
        return True

    except Exception as e:
        print(f"Erro ao criar tabela {nome_tabela}: {e}")
        return False

def adicionar_colunas_em_falta(conn, nome_tabela, df, cfg):
    try:
        cursor = conn.cursor()

        if cfg["tipo"] == "mysql":
            query = """
                SELECT COLUMN_NAME
                FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s
            """
            cursor.execute(query, (cfg["database"], nome_tabela))
        else:
            query = """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = ? AND table_name = ?
            """
            cursor.execute(query, (cfg.get("schema", "doc"), nome_tabela))

        rows = cursor.fetchall()

        colunas_existentes = set()
        for row in rows:
            if isinstance(row, dict):
                colunas_existentes.add(row.get("COLUMN_NAME") or row.get("column_name"))
            else:
                colunas_existentes.add(row[0])

        for col in df.columns:
            if col not in colunas_existentes:
                alter = f"ALTER TABLE {nome_tabela_sql(nome_tabela, cfg)} ADD COLUMN {quote_ident(col, cfg)} {tipo_texto(cfg)};"
                cursor.execute(alter)
                print(f"Coluna adicionada à tabela {nome_tabela}: {col}")

        return True

    except Exception as e:
        print(f"Erro ao adicionar colunas em falta em {nome_tabela}: {e}")
        return False

In [11]:
## Inserir DataFrame na BD
##

def inserir_dataframe_na_tabela(conn, nome_tabela, df, cfg):
    try:
        if df is None:
            raise Exception("DataFrame é None")

        cursor = conn.cursor()

        if truncate_before_insert:
            if cfg["tipo"] == "mysql":
                cursor.execute(f"TRUNCATE TABLE {nome_tabela_sql(nome_tabela, cfg)};")
            else:
                cursor.execute(f"DELETE FROM {nome_tabela_sql(nome_tabela, cfg)};")

            print(f"Tabela limpa antes da inserção: {nome_tabela}")

        if df.empty:
            print(f"Tabela {nome_tabela}: sem linhas para inserir.")
            return 0

        colunas = list(df.columns)
        placeholders = ", ".join([placeholder_sql(cfg)] * len(colunas))
        colunas_sql = ", ".join([quote_ident(c, cfg) for c in colunas])

        query = f"""
        INSERT INTO {nome_tabela_sql(nome_tabela, cfg)} ({colunas_sql})
        VALUES ({placeholders});
        """

        df = df.astype(object)

        dados = []
        for row in df.itertuples(index=False, name=None):
            linha = tuple(None if pd.isna(valor) else valor for valor in row)
            dados.append(linha)

        resultado = cursor.executemany(query, dados)

        if cfg["tipo"] == "mysql":
            inseridas = cursor.rowcount
        else:
            if isinstance(resultado, list):
                inseridas = sum(
                    item.get("rowcount", 0)
                    for item in resultado
                    if isinstance(item, dict)
                )
                if inseridas == 0:
                    inseridas = len(dados)
            else:
                inseridas = len(dados)

        print(f"{inseridas} linhas inseridas em {nome_tabela}")
        return inseridas

    except Exception as e:
        print(f"Erro ao inserir dados em {nome_tabela}: {e}")
        return 0

In [12]:
## Comparar dados da spreadsheet com a tabela original
##

COLUNAS_IGNORAR_COMPARACAO = {
    "row_id",
    "sheet_row_id",
    "created_at",
    "updated_at"
}

def preparar_dataframe_para_comparacao(df):
    if df is None:
        return None

    df = df.copy()

    # normalizar nomes das colunas
    df.columns = [limpar_nome_coluna(c) for c in df.columns]

    # remover colunas técnicas
    df = df.drop(
        columns=[c for c in COLUNAS_IGNORAR_COMPARACAO if c in df.columns],
        errors="ignore"
    )

    # trocar NaN por None
    df = df.astype(object).where(pd.notnull(df), None)

    # normalizar valores
    for col in df.columns:
        df[col] = df[col].map(lambda x: "" if x is None else str(x).strip())

    # ordenar colunas
    if "id" in df.columns:
        cols_ord = ["id"] + sorted([c for c in df.columns if c != "id"])
    else:
        cols_ord = sorted(df.columns)

    df = df[cols_ord]

    # ordenar linhas
    if len(df) > 0 and len(cols_ord) > 0:
        df = df.sort_values(by=cols_ord).reset_index(drop=True)

    return df


def ler_tabela_bd_para_dataframe(conn, nome_tabela, cfg):
    try:
        cursor = conn.cursor()
        query = f"SELECT * FROM {nome_tabela_sql(nome_tabela, cfg)};"
        cursor.execute(query)
        rows = cursor.fetchall()

        if not rows:
            return pd.DataFrame()

        df = rows_para_dataframe(rows, cursor)
        df = normalizar_dataframe(df)

        colunas_ignorar = ["row_id", "sheet_row_id", "created_at", "updated_at"]
        df = df.drop(columns=[c for c in colunas_ignorar if c in df.columns], errors="ignore")

        return df

    except Exception as e:
        print(f"Erro ao ler tabela original {nome_tabela}: {e}")
        return None


def dataframes_sao_iguais(df_sheet, df_bd, nome_folha=""):
    try:
        if df_sheet is None or df_bd is None:
            print(f"[{nome_folha}] Um dos DataFrames é None.")
            return False

        df1 = preparar_dataframe_para_comparacao(df_sheet)
        df2 = preparar_dataframe_para_comparacao(df_bd)

        # comparar colunas
        cols1 = set(df1.columns)
        cols2 = set(df2.columns)

        if cols1 != cols2:
            print(f"[{nome_folha}] Colunas só na spreadsheet: {sorted(cols1 - cols2)}")
            print(f"[{nome_folha}] Colunas só na BD: {sorted(cols2 - cols1)}")
            return False

        # garantir mesma ordem de colunas
        cols_ord = list(df1.columns)
        df2 = df2[cols_ord]

        # comparar nº de linhas
        if len(df1) != len(df2):
            print(f"[{nome_folha}] Número de linhas diferente -> spreadsheet: {len(df1)} | BD: {len(df2)}")
            return False

        # comparar conteúdo
        if not df1.equals(df2):
            print(f"[{nome_folha}] Conteúdo diferente.")
            try:
                diff = df1.compare(df2, keep_shape=False, keep_equal=False)
                print(diff.head(10))
            except:
                print("Não foi possível mostrar o diff detalhado.")
            return False

        return True

    except Exception as e:
        print(f"Erro ao comparar DataFrames ({nome_folha}): {e}")
        return False

In [13]:
## Comparar dados da spreadsheet com a tabela original
##

def importar_spreadsheet_para_bd(conn, sh, cfg):
    try:
        worksheets = sh.worksheets()
        resumo = []

        for ws in worksheets:
            nome_folha = ws.title
            print(f"\nA processar folha: {nome_folha}")

            try:
                nome_tabela_base = limpar_nome_tabela(nome_folha)
                nome_tabela_confirmed = f"{nome_tabela_base}_confirmed"

                df_sheet = worksheet_para_dataframe(ws)

                if df_sheet is None:
                    raise Exception("Falha ao converter worksheet para DataFrame")

                print(df_sheet.head())
                print("shape:", df_sheet.shape)

                df_sheet = df_sheet.drop(columns=[c for c in ["row_id", "sheet_row_id"] if c in df_sheet.columns], errors="ignore")

                df_original = ler_tabela_bd_para_dataframe(conn, nome_tabela_base, cfg)

                if df_original is None:
                    print(f"Tabela original {nome_tabela_base} não existe ou não foi possível ler. Vai ser considerado como diferente.")
                    iguais = False
                else:
                    iguais = dataframes_sao_iguais(df_sheet, df_original, nome_folha)

                if iguais:
                    print(f"A folha {nome_folha} é igual à tabela original {nome_tabela_base}. Nada será inserido em {nome_tabela_confirmed}.")
                    resumo.append({
                        "folha": nome_folha,
                        "tabela_bd": nome_tabela_confirmed,
                        "linhas_lidas": len(df_sheet),
                        "colunas": len(df_sheet.columns),
                        "linhas_inseridas": 0,
                        "estado": "igual à original - não inserido"
                    })
                    continue

                print(f"A folha {nome_folha} é diferente da tabela original {nome_tabela_base}. Vai ser inserida em {nome_tabela_confirmed}.")

                if create_tables_if_missing:
                    ok = criar_tabela_se_nao_existir(conn, nome_tabela_confirmed, df_sheet, cfg)
                    if not ok:
                        raise Exception("Não foi possível criar/verificar a tabela confirmed")

                ok_colunas = adicionar_colunas_em_falta(conn, nome_tabela_confirmed, df_sheet, cfg)
                if not ok_colunas:
                    raise Exception("Não foi possível atualizar as colunas da tabela confirmed")

                inseridas = inserir_dataframe_na_tabela(conn, nome_tabela_confirmed, df_sheet, cfg)

                resumo.append({
                    "folha": nome_folha,
                    "tabela_bd": nome_tabela_confirmed,
                    "linhas_lidas": len(df_sheet),
                    "colunas": len(df_sheet.columns),
                    "linhas_inseridas": inseridas,
                    "estado": "diferente - inserido"
                })

            except Exception as e:
                print(f"Erro ao processar folha {nome_folha}: {e}")

                resumo.append({
                    "folha": nome_folha,
                    "tabela_bd": limpar_nome_tabela(nome_folha),
                    "linhas_lidas": None,
                    "colunas": None,
                    "linhas_inseridas": 0,
                    "estado": f"erro: {e}"
                })

        df_resumo = pd.DataFrame(resumo)

        print("\nResumo da importação:\n")
        print(df_resumo)

        return df_resumo

    except Exception as e:
        print("Erro geral na importação spreadsheet -> BD:", e)
        return None

In [14]:
## Execução sequencial
##

ligacoes = []

try:
    # 1) Ligar às bases de dados configuradas
    ligacoes = ligar_bases()

    if not ligacoes:
        raise Exception("Não foi possível estabelecer ligação a nenhuma base de dados.")

    # 2) Autenticar no Google Sheets
    gc = autenticar_google()

    if gc is None:
        raise Exception("Não foi possível autenticar no Google.")

    # 3) Abrir a spreadsheet de origem
    sh = abrir_spreadsheet(gc)

    if sh is None:
        raise Exception("Não foi possível abrir a spreadsheet.")

    # 4) Importar a spreadsheet para cada base de dados
    resumos = []

    for item in ligacoes:
        conn = item["conn"]
        cfg = item["cfg"]

        print(f"\n=== A processar base: {cfg['nome']} ===")

        resumo = importar_spreadsheet_para_bd(conn, sh, cfg)

        if resumo is not None:
            resumo["base_dados"] = cfg["nome"]
            resumos.append(resumo)

    # 5) Mostrar resumo final
    if resumos:
        resumo_final = pd.concat(resumos, ignore_index=True)

        print("\n=== RESUMO FINAL ===")
        print(resumo_final)

        clts.elapt["Spreadsheet to database successful ✅"] = clts.deltat(tstart)

    else:
        print("Nenhum resumo foi produzido.")

except Exception as e:
    print("Erro durante a execução:", e)
    clts.elapt[f"Execution error ❌: {e}"] = clts.deltat(tstart)

finally:
    print("\nConnection closing....")

    for item in ligacoes:
        try:
            item["conn"].close()
            print(f"Ligação fechada: {item['cfg']['nome']}")
        except Exception as e:
            print(f"Erro ao fechar ligação: {e}")

/tmp/ipykernel_1434/1808628089.py:62: DeprecationWarning: 'db' is deprecated, use 'database'
  conn = ligar_bd(cfg)


Ligação à base de dados com sucesso! [aiven]
Ligação à base de dados com sucesso! [cratedb]
Autenticação Google com sucesso!
Spreadsheet aberta com sucesso: Confirmed

=== A processar base: aiven ===

A processar folha: catalog
  id       uri alias                                              title  \
0  1   meteo01  None  Estação meteorológica da Cobertura verde do Fo...   
1  2  qualar01  None         Estação de qualidade do ar (FORUM DA MAIA)   
2  3  qualar02  None     Estação de qualidade do ar (Rua do Património)   
3  4  qualar03  None  Estação de qualidade do ar (R. Eng. Duarte Pac...   
4  5   ruido01  None                         Medidor de ruído (Local 1)   

                                         description  \
0  Dados meteorológicos recolhidos em tempo quase...   
1          Dados de qualidade do ar recolhidos (...)   
2          Dados de qualidade do ar recolhidos (...)   
3          Dados de qualidade do ar recolhidos (...)   
4                           Medições de r